# RiverMind — waste detector v2: bottle / bag / other / vegetation

v1 (`train_detector.ipynb`) learned only "waste vs vegetation". v2 also tells **bottles** and **bags** apart, using a
second public dataset — with **no manual labelling**:

1. **Teacher:** train YOLO11n on *River Trash final* (Roboflow Universe, CC BY 4.0), which labels bottles, bags and
   other trash.
2. **Relabel:** the teacher looks at every "floating waste" box in the Mendeley turbid-river dataset (CC BY 4.0);
   where it confidently sees a bottle or bag in the same place (IoU ≥ 0.5), that box becomes bottle/bag, otherwise
   it stays "other". Vegetation stays vegetation. (Pseudo-labels: automatic, so some will be wrong.)
3. **v2:** train YOLO11n on both datasets together, evaluate on **both held-out test splits**, compare with v1 and
   the zero-shot YOLO-World baseline using one evaluator, export ONNX.

**Before you run:** upload `river-trash-final-v2-yolov11.zip` (from `data/raw/river-trash-final/` on the laptop) to
`MyDrive/rivermind/`. v1 should have finished (its weights in `MyDrive/rivermind/export/` are used for comparison).

**Run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**, choose **Select all** at the Drive
prompt. If Colab disconnects, Run all again: finished steps are skipped and training resumes.
**Results:** `MyDrive/rivermind/export/` → `yolo11n-rivermind-v2.onnx`, `.pt`, `metrics_v2.json`.

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(gpu or "NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again")

In [ ]:
%pip install -q ultralytics==8.4.43

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/rivermind")
DRIVE.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/work")
WORK.mkdir(exist_ok=True)

MENDELEY_URL = ("https://data.mendeley.com/public-files/datasets/j26w4m645z/files/"
                "fe81eec3-65c8-48c3-809b-9a9a3a359c82/file_downloaded")
MENDELEY_SHA256 = "e43166dde54f82fa9dc00adc5f3943ea7a9083f4beda1490a774fcea6dc7d8b3"
RIVER_TRASH_ZIP = DRIVE / "river-trash-final-v2-yolov11.zip"  # uploaded from the laptop
RIVER_TRASH_SHA256 = "ed8d310e2c0752bd3240441ea2393102eb3f0d83fef612d4732c431023cc5918"
SRC_M, SRC_RT, DATA = WORK / "src_mendeley", WORK / "src_rivertrash", WORK / "data_v2"
V1_WEIGHTS = DRIVE / "export" / "yolo11n-rivermind-v1.pt"

NAMES = ["bottle", "bag", "other", "vegetation"]  # CONTRACT.md classes; vegetation is detected but never counted
TEACHER, RUN_NAME = "yolo11n-rivertrash-teacher", "yolo11n-rivermind-v2"
TEACHER_IMGSZ, TEACHER_EPOCHS = 640, 40  # River Trash photos are mostly <= 640 px
IMGSZ, EPOCHS, PATIENCE = 960, 60, 15
RELABEL_IOU, RELABEL_CONF = 0.5, 0.35
COUNT_CONF = 0.25  # confidence used when counting items per image for the count-error metric
MAX_SIDE, SEED, FRACTION = 1920, 0, 1.0
RUN_BASELINE = True

In [ ]:
import hashlib
import subprocess


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


mendeley_zip = WORK / "turbid-rivers-v2.zip"
if not mendeley_zip.exists():
    print("downloading 1.76 GB from Mendeley ...")
    subprocess.run(["curl", "-L", "--fail", "-sS", "-A", "Mozilla/5.0", "-o", str(mendeley_zip), MENDELEY_URL],
                   check=True)
assert sha256(mendeley_zip) == MENDELEY_SHA256, "Mendeley checksum mismatch: delete /content/work and Run all again"
assert RIVER_TRASH_ZIP.exists(), f"upload river-trash-final-v2-yolov11.zip to {DRIVE} first"
assert sha256(RIVER_TRASH_ZIP) == RIVER_TRASH_SHA256, "River Trash zip checksum mismatch: re-upload it"
print("both datasets verified")

In [ ]:
import os
import shutil
import zipfile

import yaml
from PIL import Image

for src, archive in ((SRC_M, mendeley_zip), (SRC_RT, RIVER_TRASH_ZIP)):
    if not src.exists():
        with zipfile.ZipFile(archive) as z:
            z.extractall(src)
m_root = SRC_M / "Floating Waste and River Vegetation Dataset"

# River Trash final ids -> ours (bottle 0, bag 1, other 2, vegetation 3). oil-spil is a slick, not an object: dropped.
RT_NAMES = ["branch", "cardboard", "clustered-trash", "cup", "metal-can", "oil-spil", "plastic-bag", "plastic-bottle",
            "plastic-container", "styrofoam"]
RT_MAP = {7: 0, 6: 1, 1: 2, 2: 2, 3: 2, 4: 2, 8: 2, 9: 2, 0: 3}
assert yaml.safe_load((SRC_RT / "data.yaml").read_text())["names"] == RT_NAMES, "unexpected River Trash classes"


def read_labels(path):
    rows = []
    for line in path.read_text().splitlines() if path.exists() else []:
        parts = line.split()
        if len(parts) == 5:
            rows.append((int(parts[0]), *parts[1:]))
    return rows


def write_labels(path, rows):
    path.write_text("".join(" ".join(map(str, row)) + "\n" for row in rows))


def copy_image(src, dst):
    im = Image.open(src)
    if im.getexif().get(0x0112, 1) != 1:  # rotation tag: boxes may not line up with the pixels
        return False
    if max(im.size) > MAX_SIDE:
        im.thumbnail((MAX_SIDE, MAX_SIDE))  # boxes are normalised, labels stay valid
        im.convert("RGB").save(dst, quality=95)
    else:
        shutil.copy(src, dst)
    return True


def link(src, dst):
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy(src, dst)


shutil.rmtree(DATA, ignore_errors=True)
stats = {}
for split, rt_split in (("train", "train"), ("val", "valid"), ("test", "test")):
    for kind in ("rt", "m_raw"):
        (DATA / kind / split / "images").mkdir(parents=True)
        (DATA / kind / split / "labels").mkdir(parents=True)
    n_rt = n_m = 0
    for img in sorted((SRC_RT / rt_split / "images").iterdir()):
        dst = DATA / "rt" / split / "images" / f"{img.stem}.jpg"
        if img.suffix.lower() not in (".jpg", ".jpeg", ".png") or not copy_image(img, dst):
            continue
        rows = [(RT_MAP[c], *xywh) for c, *xywh in read_labels(SRC_RT / rt_split / "labels" / f"{img.stem}.txt")
                if c in RT_MAP]
        write_labels(DATA / "rt" / split / "labels" / f"{img.stem}.txt", rows)
        n_rt += 1
    for img in sorted((m_root / split / "images").glob("*.jpg")):
        if copy_image(img, DATA / "m_raw" / split / "images" / img.name):
            shutil.copy(m_root / split / "labels" / f"{img.stem}.txt", DATA / "m_raw" / split / "labels")
            n_m += 1
    stats[split] = {"river_trash": n_rt, "mendeley": n_m}
print("images per split:", stats)


def write_yaml(name, root, train, val, names):
    path = DATA / name
    path.write_text(f"path: {root}\ntrain: {train}\nval: {val}\nnames:\n"
                    + "".join(f"  {i}: {n}\n" for i, n in enumerate(names)))
    return path


teacher_yaml = write_yaml("teacher.yaml", DATA / "rt", "train/images", "val/images", NAMES)
rt_test_yaml = write_yaml("rt_test.yaml", DATA / "rt", "train/images", "test/images", NAMES)  # val = test split
v2_yaml = write_yaml("v2.yaml", DATA / "v2", "train/images", "val/images", NAMES)

In [ ]:
import gc

import torch
from ultralytics import YOLO

runs = DRIVE / "runs"


def train(name, cache="ram", **kwargs):
    last = runs / name / "weights" / "last.pt"
    if last.exists():  # Colab disconnected mid-run: resume from the last saved epoch
        try:
            YOLO(str(last)).train(resume=True)
        except AssertionError as finished:  # the run already completed; nothing to resume
            print(finished)
    else:
        YOLO("yolo11n.pt").train(project=str(runs), name=name, exist_ok=True, seed=SEED, deterministic=True,
                                 workers=2, fraction=FRACTION, batch=16, cache=cache, **kwargs)
    gc.collect()  # drop the trainer and its image cache before the next step
    torch.cuda.empty_cache()
    return YOLO(str(runs / name / "weights" / "best.pt"))


teacher = train(TEACHER, data=str(teacher_yaml), imgsz=TEACHER_IMGSZ, epochs=TEACHER_EPOCHS, patience=10)

In [ ]:
import numpy as np


def iou(a, b):
    # IoU between every box in a (n, 4) and b (m, 4), xyxy
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(axis=2)
    area_a = (a[:, 2:] - a[:, :2]).prod(axis=1)
    area_b = (b[:, 2:] - b[:, :2]).prod(axis=1)
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-9)


def to_xyxy(cx, cy, w, h):
    cx, cy, w, h = map(float, (cx, cy, w, h))
    return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


shutil.rmtree(DATA / "v2", ignore_errors=True)
relabelled = dict.fromkeys(NAMES, 0)
for split in ("train", "val"):
    out_img, out_lbl = DATA / "v2" / split / "images", DATA / "v2" / split / "labels"
    out_img.mkdir(parents=True)
    out_lbl.mkdir(parents=True)
    for img in sorted((DATA / "rt" / split / "images").iterdir()):  # River Trash: real labels
        link(img, out_img / f"rt_{img.name}")
        shutil.copy(DATA / "rt" / split / "labels" / f"{img.stem}.txt", out_lbl / f"rt_{img.stem}.txt")
    imgs = sorted((DATA / "m_raw" / split / "images").glob("*.jpg"))  # Mendeley: teacher relabels "floating waste"
    for img in imgs:  # one image at a time: a big list kept every decoded photo in RAM and crashed Colab
        result = teacher.predict(str(img), imgsz=IMGSZ, conf=RELABEL_CONF, verbose=False)[0]
        boxes = result.boxes.xyxyn.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        keep = np.isin(classes, (0, 1))  # only bottle / bag predictions can relabel a box
        boxes, classes = boxes[keep], classes[keep]
        rows = []
        for c, *xywh in read_labels(DATA / "m_raw" / split / "labels" / f"{img.stem}.txt"):
            new = 3 if c == 1 else 2  # Mendeley: 0 floating_waste -> other (default), 1 river_vegetation -> vegetation
            if c == 0 and len(boxes):
                overlaps = iou(np.array([to_xyxy(*xywh)]), boxes)[0]
                if overlaps.max() >= RELABEL_IOU:
                    new = int(classes[overlaps.argmax()])
            relabelled[NAMES[new]] += 1
            rows.append((new, *xywh))
        link(img, out_img / f"m_{img.name}")
        write_labels(out_lbl / f"m_{img.stem}.txt", rows)
print("Mendeley boxes after relabelling (train + val):", relabelled)

In [ ]:
# no RAM cache here: ~2,300 photos at 960 px would not fit next to everything else in Colab's 12 GB
student = train(RUN_NAME, cache=False, data=str(v2_yaml), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE)

## Evaluation on both held-out test splits
- **Per class (bottle / bag / other / vegetation):** Ultralytics val on the River Trash test split (real labels).
- **Same yardstick for v1, v2 and the zero-shot baseline:** our evaluator collapses bottle/bag/other into "waste" so
  every model can be scored on both test splits, and also reports **count error** (mean absolute difference between
  items counted at confidence ≥ 0.25 and the true number of waste items per image) — the number verification uses.
  Mendeley test labels are the authors' originals, never pseudo-labels.

In [ ]:
import json

from ultralytics.utils.metrics import compute_ap

EXPORT = DRIVE / "export"
EXPORT.mkdir(exist_ok=True)


def collapsed_eval(model, split_dir, gt_map, pred_map):
    imgs = sorted((split_dir / "images").iterdir())
    records, n_gt = {"waste": [], "vegetation": []}, {"waste": 0, "vegetation": 0}
    count_err, gt_counts = [], []
    for img in imgs:
        result = model.predict(str(img), imgsz=IMGSZ, conf=0.001, iou=0.7, max_det=300, verbose=False)[0]
        gts = [(gt_map[c], to_xyxy(*xywh)) for c, *xywh in read_labels(split_dir / "labels" / f"{img.stem}.txt")]
        boxes = result.boxes.xyxyn.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        order = np.argsort(-confs)  # most confident first
        for name in records:
            gt = np.array([b for c, b in gts if c == name]).reshape(-1, 4)
            n_gt[name] += len(gt)
            matched = np.zeros(len(gt), bool)
            mine = [i for i in order if pred_map.get(int(classes[i])) == name]
            for i in mine:
                hit = False
                if len(gt):
                    overlaps = iou(boxes[i:i + 1], gt)[0]
                    overlaps[matched] = -1
                    if overlaps.max() >= 0.5:
                        matched[overlaps.argmax()] = hit = True
                records[name].append((float(confs[i]), hit))
            if name == "waste":
                count_err.append(abs(sum(confs[i] >= COUNT_CONF for i in mine) - len(gt)))
                gt_counts.append(len(gt))
    out = {"images": len(imgs)}
    for name, recs in records.items():
        recs.sort(key=lambda r: -r[0])
        tp = np.cumsum([r[1] for r in recs]) if recs else np.zeros(0)
        fp = np.cumsum([not r[1] for r in recs]) if recs else np.zeros(0)
        ap = compute_ap(tp / n_gt[name], tp / np.maximum(tp + fp, 1e-9))[0] if recs and n_gt[name] else 0.0
        out[f"{name}_mAP50"], out[f"{name}_boxes"] = round(float(ap), 4), n_gt[name]
    out.update(count_mae=round(float(np.mean(count_err)), 2), mean_true_count=round(float(np.mean(gt_counts)), 2),
               count_conf=COUNT_CONF)
    return out


TESTS = {"mendeley_test": (DATA / "m_raw" / "test", {0: "waste", 1: "vegetation"}),
         "river_trash_test": (DATA / "rt" / "test", {0: "waste", 1: "waste", 2: "waste", 3: "vegetation"})}
models = {"v2 (" + RUN_NAME + ")": (student, {0: "waste", 1: "waste", 2: "waste", 3: "vegetation"})}
if V1_WEIGHTS.exists():
    models["v1 (yolo11n-rivermind-v1)"] = (YOLO(str(V1_WEIGHTS)), {0: "waste", 1: "vegetation"})
if RUN_BASELINE:
    try:
        world = YOLO("yolov8s-worldv2.pt")
        world.set_classes(["floating trash", "aquatic plants"])
        models["zero-shot yolov8s-worldv2"] = (world, {0: "waste", 1: "vegetation"})
    except Exception as exc:  # optional comparison only
        print("zero-shot baseline skipped:", exc)

per_class = student.val(data=str(rt_test_yaml), split="val", imgsz=IMGSZ, batch=16, project=str(runs),
                        name=f"{RUN_NAME}-rt-test", exist_ok=True)
results = {
    "v2_per_class_mAP50_river_trash_test": {NAMES[int(c)]: round(float(v), 4) for c, v in
                                            zip(per_class.box.ap_class_index, per_class.box.ap50, strict=True)},
    "relabelled_mendeley_boxes": relabelled,
    "note": "collapsed = bottle/bag/other scored as one 'waste' class; greedy matching at IoU 0.5; our evaluator, "
            "not identical to Ultralytics val",
    "collapsed": {model_name: {test: collapsed_eval(m, d, gt, pm) for test, (d, gt) in TESTS.items()}
                  for model_name, (m, pm) in models.items()},
}
(EXPORT / "metrics_v2.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))

In [ ]:
onnx_path = Path(student.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False))
size_mb = onnx_path.stat().st_size / 1e6
assert size_mb <= 25, f"ONNX is {size_mb:.1f} MB, over the 25 MB contract cap"
shutil.copy(onnx_path, EXPORT / f"{RUN_NAME}.onnx")
shutil.copy(runs / RUN_NAME / "weights" / "best.pt", EXPORT / f"{RUN_NAME}.pt")
shutil.copy(runs / RUN_NAME / "results.csv", EXPORT / "results_v2.csv")
print(f"exported {RUN_NAME}.onnx ({size_mb:.1f} MB) to {EXPORT}")